# Computer Vision Track · CV Masterclass 03: Vision Transformers (ViT)

Welcome to Notebook 03. For a decade, Convolutional Neural Networks (CNN) ruled Computer Vision unchallenged. 

Then, taking inspiration from the massive success of NLP Large Language Models, researchers asked: *"What if we just treat an image exactly like a sentence of words, and feed it into a standard Transformer?"*

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"CNNs possess 'Inductive Bias'—a hardcoded mathematical assumption that pixels near each other are highly correlated. Vision Transformers have absolutely zero Inductive Bias. Why does this lack of mathematical assumption mean ViTs require 10x more training data than CNNs to achieve the same accuracy, but eventually allows them to drastically outperform CNNs when scaled to extreme datasets?"*

---

1. **Self-Attention Math:** The $O(N^2)$ foundation.
2. **FlashAttention:** Efficiency via Hardware-Awareness.
3. **The Patching Paradigm:** Overcoming pixel-level complexity.
4. **Position Embeddings:** Solving spatial blindness (1D vs RoPE).
5. **The `[CLS]` Token:** Sequence to classification mapping.
6. **Inductive Bias & Scaling Laws:** Why Transformers win with more data.
7. **Swin Transformers:** Hierarchical Vision via shifted windows.
8. **CLIP:** Zero-Shot multimodal contrastive learning.
9. **Vision Mamba (Vim):** Linear-time $O(N)$ global modeling.
10. **MAE & DeiT:** Self-Supervision and Distillation.
11. **Deep Socratic Synthesis.**


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [1. Self-Attention Math](#1-self-attention-math)
* [2. FlashAttention: Breaking the Memory Wall](#2-flashattention-breaking-the-memory-wall)
* [3. The Patching Paradigm](#3-the-patching-paradigm)
* [4. Position Embeddings (Absolute & RoPE)](#4-position-embeddings-absolute-rope)
* [5. The `[CLS]` Token](#5-the-cls-token)
* [6. Inductive Bias & Scaling Laws](#6-inductive-bias-scaling-laws)
* [7. Swin Transformers (Hierarchical Attention)](#7-swin-transformers-hierarchical-attention)
* [8. CLIP (Contrastive Language-Image)](#8-clip-contrastive-language-image)
* [9. Vision Mamba (Vim) - The O(N) Alternative](#9-vision-mamba-vim-the-on-alternative)
* [10. MAE & DeiT: Training Tactics](#10-mae-deit-training-tactics)
* [11. 🎓 Deep Socratic Synthesis](#11-deep-socratic-synthesis)


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Assumes Vision Transformers are just NLP models shoved into pictures because transformers are trendy.

**Senior:** Understands that ViTs lack the 'inductive bias' of translation invariance found in CNNs, making them highly data-hungry. However, their global attention allows them to model distal relationships immediately, giving them a much higher performance ceiling at massive scale.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Assumes Vision Transformers are just NLP models shoved into pictures because transformers are trendy.

**Senior:** Understands that ViTs lack the 'inductive bias' of translation invariance found in CNNs, making them highly data-hungry. However, their global attention allows them to model distal relationships immediately, giving them a much higher performance ceiling at massive scale.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Assumes Vision Transformers are just NLP models shoved into pictures because transformers are trendy.

**Senior:** Understands that ViTs lack the 'inductive bias' of translation invariance found in CNNs, making them highly data-hungry. However, their global attention allows them to model distal relationships immediately, giving them a much higher performance ceiling at massive scale.

---


## 1. Self-Attention Math

How does a patch gather context from all other patches? By calculating Query ($Q$), Key ($K$), and Value ($V$) projections and using **Scaled Dot-Product Attention**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) \cdot V$$


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

dummy = torch.randn(1, 196, 768)
mha = MultiHeadAttention(dim=768, num_heads=12)
print(f"Attention Output: {mha(dummy).shape}")


## 2. FlashAttention: Breaking the Memory Wall

Standard attention materializes $N \times N$ maps. FlashAttention uses tiling and Online Softmax to compute results in fast SRAM memory, avoiding the quadratic HBM memory bottleneck. Global attention on $1024\times1024$ images is now possible.

## 3. The Patching Paradigm

Transformers can't process $50,176$ pixels ($224^2$). We slice the image into **Patches** (e.g., $16\times16$). 
A $224\times224$ image yields exactly $14\times14 = \mathbf{196}$ tokens. This brings $N$ back to a range where $O(N^2)$ is computationally stable.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=16, in_ch=3, embed_dim=768):
        super().__init__()
        # The convolution is the standard trick to slice and project in one step
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # (B, 3, 224, 224) -> (B, 768, 14, 14)
        x = self.proj(x)
        # (B, 768, 14, 14) -> (B, 196, 768)
        x = x.flatten(2).transpose(1, 2)
        return x

dummy_img = torch.randn(1, 3, 224, 224)
pe = PatchEmbedding()
print(f"Patch Token Sequence: {pe(dummy_img).shape}")


## 4. Position Embeddings (Absolute & RoPE)

Attention has no spatial awareness. We must add position info:
- **Absolute**: $N$ learnable vectors added to tokens. Fails if resolution changes.
- **RoPE (Rotary)**: Rotates Query/Key vectors based on $(x, y)$ coordinates. Natively supports varying resolutions and learns relative distance logic.

## 5. The `[CLS]` Token

A dummy token injected at index 0. It gathers global context from all $N$ image patches. At the head, we discard the image tokens and classify only the `[CLS]` vector.

## 6. Inductive Bias & Scaling Laws

CNNs have a strong local bias. ViTs have none. On small datasets (ImageNet-1K), CNNs win. On massive datasets (JFT-300M), ViTs see "the whole picture" and achieve superhuman performance.

## 7. Swin Transformers (Hierarchical Attention)

ViT uses global attention ($O(N^2)$). Swin uses **Windowed Attention** ($O(N)$) on $7\times7$ windows and **Shifted Windows** to allow cross-window communication. This brought back the hierarchical nature of CNNs to Transformers.

## 8. CLIP (Contrastive Language-Image)

OpenAI CLIP trained a ViT Image Encoder and a Text Encoder simultaneously. By maximizing the dot product of matching (image, text) pairs, CLIP learned a semantic space that enables **Zero-Shot Classification**.

## 9. Vision Mamba (Vim) - The O(N) Alternative

Mamba uses a Selective State Space Model instead of attention. It models sequences in linear time. Vim flattens the image and performs bidirectional scanning, offering a global receptive field with zero quadratic bottleneck.

## 10. MAE & DeiT: Training Tactics

ViTs are data-hungry. Two techniques solved this:
- **MAE (Masked Autoencoders)**: Mask $75\%$ of patches. Force the network to reconstruct the missing pixels. This self-supervised task creates extremely robust features.
- **DeiT (Data-Efficient)**: Add a **Distillation Token**. The ViT learns from a CNN Teacher (e.g., RegNet). This allows training ViTs on ImageNet-1K without massive proprietary datasets.

In [ ]:
import timm

def load_sota_vit():
    # timm is the industry standard for SOTA Vision Transformers
    model = timm.create_model('vit_base_patch16_224', pretrained=True)
    return model

print("SOTA loading logic with 'timm' demonstrated.")


---
## ✅ Knowledge Check

### Q1: What does the 'CLS token' do in a Vision Transformer?
<details><summary>👉 Answer</summary>

It is a learned embedding appended to the sequence of image patches. Its final state is used as the global image representation for the classification head.
</details>

### Q2: Why do standard ViTs require massive datasets compared to ResNets?
<details><summary>👉 Answer</summary>

CNNs have built-in inductive biases (local neighborhoods, translation invariance). Transformers have no such biases and must learn that pixels are spatially related purely from raw data and positional embeddings.
</details>

---
## 🔨 Practical Practice

### Exercise 1
Patch Extraction: Use `einops.rearrange` or PyTorch `unfold` to slice a 224x224 image tensor into 16x16 flattened patches.

### Exercise 2
Positional Encoding: Implement 1D learnable positional embeddings and add them to a batch of image patches.


## 11. 🎓 Deep Socratic Synthesis

**Q:** *If we have a $224 \times 224$ image and use $16 \times 16$ patches, we have 196 tokens. If we use $8 \times 8$ patches, we have 784 tokens. Why does doubling the resolution (4x tokens) increase the Attention FLOPs by **16x**, and how does Swin's windowing specifically decouple this?*

**Ponder deeply:** In global attention, every token looks at every other token ($N \times N$). If $N$ increases by $4\times$, the matrix operations grow by $4^2 = 16\times$. Swin solves this by restricting attention to a fixed-size window (e.g., $7\times7$). When resolution grows, we simply have *more* windows, but each window still performs the same fixed computation ($49 \times 49$), leading to a **Linear** $O(N)$ scaling instead of quadratic.